<a href="https://colab.research.google.com/github/thisishasan/speech_processing/blob/main/03_training_model_llava.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.6 MB/s eta 0:00:00


In [2]:
from pathlib import Path

train_file = Path(
    "/content/drive/MyDrive/01_speech_processing/"
    "sp_exam_project/dataset/llava_dataset/llava_train.json"
)

print("Exists:", train_file.exists())
print("Size:", train_file.stat().st_size)

Exists: True
Size: 2533343


In [2]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
import json
import os
from pathlib import Path

import torch
from google.colab import drive
from PIL import Image
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    Trainer,
    TrainingArguments,
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

drive.mount("/content/drive")

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset"
)
DATASET_DIR = DRIVE_ROOT / "llava_dataset"
IMAGE_DIR = DRIVE_ROOT / "images"
OUTPUT_DIR = DRIVE_ROOT / "hf_llava_checkpoints"
TRAIN_FILE = DATASET_DIR / "llava_train.json"

# Hugging Face LLaVA checkpoint. This is separate from the old original-LLaVA
# repository and does not require changing repository source files.
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

NUM_EPOCHS = 2
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.03
MAX_LENGTH = 2048
SAVE_STEPS = 500
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 2
NUM_WORKERS = 2

# The A100 supports bfloat16. Full BF16 base-model loading plus LoRA fits an
# A100 80 GB for this 7B model. Set to False only if your GPU has less memory.
USE_BF16 = True


# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------

class FlickrVQADataset(Dataset):
    def __init__(self, json_path, image_dir):
        with open(json_path, "r", encoding="utf-8") as file:
            self.records = json.load(file)
        self.image_dir = Path(image_dir)

        if not isinstance(self.records, list) or not self.records:
            raise ValueError("The training JSON must contain at least one record.")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        filename = str(record["image"])
        image_path = self.image_dir / filename

        if not image_path.exists():
            # Support records containing an absolute image path as a fallback.
            stored_path = Path(filename)
            if stored_path.is_absolute() and stored_path.exists():
                image_path = stored_path
            else:
                raise FileNotFoundError(f"Image not found: {image_path}")

        image = Image.open(image_path).convert("RGB")
        question = " ".join(str(record["conversations"][0]["value"]).split())
        answer = " ".join(str(record["conversations"][1]["value"]).split())

        # The HF LLaVA 1.5 prompt format uses USER/ASSISTANT turns and one
        # image placeholder. The answer is included during supervised training.
        question = question.replace("<image>", "").strip()
        prompt = f"USER: <image>\n{question} ASSISTANT: {answer}"

        return {"image": image, "prompt": prompt}


def make_collator(processor):
    def collate(examples):
        images = [example["image"] for example in examples]
        prompts = [example["prompt"] for example in examples]

        batch = processor(
            images=images,
            text=prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100

        # Do not calculate loss on visual placeholder tokens.
        image_token_id = getattr(processor, "image_token_id", None)
        if image_token_id is None:
            image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
        if image_token_id is not None and image_token_id >= 0:
            labels[labels == image_token_id] = -100

        batch["labels"] = labels
        return batch

    return collate


# -----------------------------------------------------------------------------
# Model and LoRA training
# -----------------------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError("An NVIDIA GPU is required for Hugging Face LLaVA training.")

if not TRAIN_FILE.exists():
    raise FileNotFoundError(f"Training file not found: {TRAIN_FILE}")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image directory not found: {IMAGE_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_dataset = FlickrVQADataset(TRAIN_FILE, IMAGE_DIR)

dtype = torch.bfloat16 if USE_BF16 else torch.float16
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="eager",
)
model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

steps_per_epoch = (
    len(train_dataset) + PER_DEVICE_BATCH_SIZE - 1
) // PER_DEVICE_BATCH_SIZE
optimizer_steps = (
    steps_per_epoch + GRADIENT_ACCUMULATION_STEPS - 1
) // GRADIENT_ACCUMULATION_STEPS
total_steps = optimizer_steps * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    dataloader_num_workers=NUM_WORKERS,
    report_to="none",
)

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_ID)
print("Training records:", f"{len(train_dataset):,}")
print("Optimizer steps:", f"{total_steps:,}")
print("Warm-up steps:", warmup_steps)
print("Output directory:", OUTPUT_DIR)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=make_collator(processor),
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "final_adapter"))
processor.save_pretrained(str(OUTPUT_DIR / "final_adapter"))

print("\\nHugging Face LLaVA fine-tuning completed.")
print("Final adapter saved to:", OUTPUT_DIR / "final_adapter")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

trainable params: 19,136,512 || all params: 7,082,563,584 || trainable%: 0.2702
GPU: NVIDIA A100-SXM4-80GB
Model: llava-hf/llava-1.5-7b-hf
Training records: 9,294
Optimizer steps: 1,162
Warm-up steps: 34
Output directory: /content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/hf_llava_checkpoints


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,6.294406
20,6.183001
30,5.766253
40,4.928094
50,3.893380
60,2.870162
70,2.284370
80,1.981004
90,1.573647
100,1.350844


\nHugging Face LLaVA fine-tuning completed.
Final adapter saved to: /content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/hf_llava_checkpoints/final_adapter


In [6]:
!ls -la $OUTPUT_DIR/final_adapter

total 78370
drwx------ 2 root root     4096 Sep 13 10:10 .
drwx------ 5 root root     4096 Sep 13 10:10 ..
-rw------- 1 root root     1108 Sep 13 10:10 adapter_config.json
-rw------- 1 root root 76606728 Sep 13 10:10 adapter_model.safetensors
-rw------- 1 root root      674 Sep 13 10:10 chat_template.jinja
-rw------- 1 root root      713 Sep 13 10:10 processor_config.json
-rw------- 1 root root     5202 Sep 13 10:10 README.md
-rw------- 1 root root      586 Sep 13 10:10 tokenizer_config.json
-rw------- 1 root root  3619380 Sep 13 10:10 tokenizer.json
-rw------- 1 root root     5265 Sep 13 10:10 training_args.bin
